# فاز ۱ — EDA (Bank Marketing)  


**Dataset:** Bank Marketing (Deposit Subscription Prediction)

```bash
python -m src.data.download_bank_marketing
```



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.load_bank_marketing import load_raw

df = load_raw()
df.head()

In [ ]:
print('shape:', df.shape)
df.info()

## 1) بررسی برچسب (Target) و عدم‌توازن کلاس


In [ ]:
target_col = 'y'
df[target_col] = df[target_col].astype(str)
vc = df[target_col].value_counts(dropna=False)
vc

In [ ]:
# Plot 1: توزیع برچسب
plt.figure()
vc.plot(kind='bar')
plt.title('Target distribution (y)')
plt.xlabel('class')
plt.ylabel('count')
plt.tight_layout()
plt.show()

## 2) شناسایی ویژگی‌های عددی و دسته‌ای + وضعیت مقدارهای Unknown/NaN

در Bank Marketing مقدارهای `unknown` معمولاً نقش Missing را دارند و باید در Feature Engineering تصمیم‌گیری شوند.


In [ ]:
X = df.drop(columns=[target_col])
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
num_cols, cat_cols, len(num_cols), len(cat_cols)

In [ ]:
# درصد NaN در هر ستون
nan_rate = X.isna().mean().sort_values(ascending=False)
nan_rate.head(20)

In [ ]:
# درصد 'unknown' در ستون‌های دسته‌ای
unknown_rate = {}
for c in cat_cols:
    unknown_rate[c] = (X[c].astype(str).str.lower() == 'unknown').mean()
unknown_rate = pd.Series(unknown_rate).sort_values(ascending=False)
unknown_rate.head(20)

In [ ]:
# Plot 2: بیشترین نرخ unknown در ستون‌های دسته‌ای
plt.figure(figsize=(8,4))
unknown_rate.head(12).plot(kind='bar')
plt.title('Top categorical columns by UNKNOWN rate')
plt.ylabel('rate')
plt.tight_layout()
plt.show()

## 3) توزیع ویژگی‌های عددی (Histogram)

هدف: درک skew/outlier و نیاز به scaling / transform.


In [ ]:
# Plot 3: چند هیستوگرام عددی
cols = num_cols[:6]  # اگر زیاد است فقط چندتا
X[cols].hist(bins=30, figsize=(10,6))
plt.suptitle('Numeric feature histograms')
plt.tight_layout()
plt.show()

## 4) مقایسه‌ی توزیع عددی‌ها بین کلاس‌ها (Boxplot)

هدف: پیدا کردن ویژگی‌هایی که بین کلاس‌ها جداشدگی دارند.


In [ ]:
# Plot 4: boxplot یک/چند ویژگی مهم عددی برحسب کلاس
plt.figure(figsize=(10,4))
box_cols = [c for c in ['age','balance','duration','campaign'] if c in X.columns]
df_box = df[box_cols + [target_col]].copy()
df_box.boxplot(column=box_cols, by=target_col)
plt.suptitle('')
plt.title('Numeric features by class')
plt.tight_layout()
plt.show()

## 5) همبستگی ویژگی‌های عددی (Correlation Heatmap)

هدف: تشخیص ویژگی‌های همبسته و احتمال نیاز به regularization یا انتخاب ویژگی.


In [ ]:
corr = df[num_cols].corr(numeric_only=True)
plt.figure(figsize=(8,6))
plt.imshow(corr, aspect='auto')
plt.colorbar()
plt.title('Correlation matrix (numeric features)')
plt.tight_layout()
plt.show()

## 6) یک نمای دوبعدی (Scatter) برای دیدن separability


In [ ]:
# Plot 6: scatter (age vs balance) رنگی برحسب کلاس (اگر موجود)
x1, x2 = ('age','balance')
if x1 in df.columns and x2 in df.columns:
    plt.figure(figsize=(6,4))
    for cls, mk in [(df[target_col].unique()[0], 'o'), (df[target_col].unique()[1], 'x')]:
        sub = df[df[target_col] == cls]
        plt.scatter(sub[x1], sub[x2], s=10, alpha=0.4, marker=mk, label=str(cls))
    plt.xlabel(x1)
    plt.ylabel(x2)
    plt.title(f'Scatter: {x1} vs {x2} by class')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Columns not found for scatter plot')

## 7) نرخ مثبت (subscription=yes) در گروه‌های دسته‌ای

این نمودارها کمک می‌کند Feature Engineering مثل Target Encoding / One-Hot / grouping تصمیم‌گیری شود.


In [ ]:
# Plot 7: نرخ مثبت در یک ویژگی دسته‌ای (مثلاً job)
cat = 'job' if 'job' in df.columns else (cat_cols[0] if len(cat_cols)>0 else None)
if cat is not None:
    tmp = df[[cat, target_col]].copy()
    tmp['is_yes'] = (tmp[target_col].str.lower() == 'yes').astype(int)
    rates = tmp.groupby(cat)['is_yes'].mean().sort_values(ascending=False)
    plt.figure(figsize=(10,4))
    rates.head(15).plot(kind='bar')
    plt.title(f'Positive rate by {cat} (top 15)')
    plt.ylabel('P(y=yes)')
    plt.tight_layout()
    plt.show()
else:
    print('No categorical columns found')